# 🚀 Smart Stray Rescue: Qwen2-VL & FastAPI Backend
This notebook runs the **Full Vision-Language Backend** on Google Colab.

### 📦 Components:
1.  **Qwen2-VL Model**: For advanced animal classification and emotion recognition.
2.  **FastAPI Backend**: Provides a robust, OpenAI-compatible `/v1/chat/completions` endpoint.
3.  **LocalTunnel**: Exposes the port for your Rescue frontend.

---

In [ ]:
# @title 1. Install Dependencies
!pip install -q transformers accelerate qwen-vl-utils fastapi uvicorn python-multipart pydantic pillow numpy httpx
!pip install -q flash-attn --no-build-isolation
!npm install -g localtunnel

In [ ]:
# @title 2. Load Qwen2-VL Model
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"Loading {MODEL_ID}...")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model loaded successfully!")

In [ ]:
# @title 3. Start Vision API
import os
import json
import asyncio
import base64
import re
from io import BytesIO
from PIL import Image
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from qwen_vl_utils import process_vision_info
import uvicorn
import threading

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
async def root():
    return {"status": "SUCCESS", "message": "Rescue Vision Server is Online", "model": "Qwen2-VL-2B"}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    data = await request.json()
    messages = data.get("messages", [])

    processed_messages = []
    for msg in messages:
        new_content = []
        for item in msg.get("content", []):
            if item.get("type") == "image_url":
                url = item["image_url"]["url"]
                if "," in url: url = url.split(",")[1]
                image_data = base64.b64decode(url)
                image = Image.open(BytesIO(image_data)).convert("RGB")
                new_content.append({"type": "image", "image": image})
            else:
                new_content.append(item)
        processed_messages.append({"role": msg["role"], "content": new_content})

    text = processor.apply_chat_template(processed_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(processed_messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    inputs = inputs.to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=150)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

    print(f"Vision Analysis Result: {output_text}")
    return {
        "choices": [
            {
                "message": {
                    "role": "assistant",
                    "content": output_text
                }
            }
        ]
    }

def run_server():
    print("Starting Vision API on port 8000...")
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

In [ ]:
# @title 4. Start Tunnel & Get URL
print("\n--- ACTION REQUIRED ---")
print("1. Use the URL below in your Rescue Website settings.")
print("2. If asked for a password, use this IP: ", end="")
!curl -s ipv4.icanhazip.com
print("------------------------\n")
!npx localtunnel --port 8000